In [5]:
1+1

2

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
## Determenistic approch for guardrails

def determenistic_guard_rail(text:str)->bool:
    """Returns True if content is blocked"""
    banned_words = ["hack","malware","exploit"]
    return any(kw in text.lower() for kw in banned_words)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]
print("===Determenistic guardrail approach===")
for inp in test_inputs:
    blocked = determenistic_guard_rail(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}:{inp}")

===Determenistic guardrail approach===
Blocked:How do I hack into a database?
Allowed:What is the capital of France?
Blocked:Explain how malware spreads


In [8]:
from langchain_groq import ChatGroq

def model_based_guardrail(text:str)->str:
    """ uses a llm for the safety,Returns SAFE OR UNSAFE"""
    model = ChatGroq(model = "llama-3.3-70b-versatile",temperature = 0)
    prompt = f"""is the following input is safe to process? Reply with
    SAFE OR UNSAFE
    Input:{text}"""
    
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("===Model Based Guardrail===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE"if "UNSAFE" in verdict else "SAFE"
    print(f"{status}:{inp}")

    

===Model Based Guardrail===
UNSAFE:How do I hack into a database?
SAFE:What is the capital of France?
SAFE:Explain how malware spreads


In [9]:
## Langchain has inbuilt middleware PII personally identifiable information

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool

@tool
def customer_records(query:str)-> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [customer_records],
    middleware=[
        PIIMiddleware(
            "email",
            strategy='redact',
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        ),
    ],
)
print("Agent with PII middleware created sucessfully")

Agent with PII middleware created sucessfully


In [10]:
response = agent.invoke({"messages":[{"role":"user","content":"My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"}]})
response["messages"][-1].content

"I've located your customer record. Is there something specific you'd like to know or change regarding your account?"

In [11]:
response

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='55f83eee-51ca-4b26-aa3c-d5faab6f9f46'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'd18rrm8kj', 'function': {'arguments': '{"query":"[REDACTED_EMAIL] and card ****-****-****-5100"}', 'name': 'customer_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 236, 'total_tokens': 266, 'completion_time': 0.090519924, 'completion_tokens_details': None, 'prompt_time': 0.012101628, 'prompt_tokens_details': None, 'queue_time': 0.051777516, 'total_time': 0.102621552}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdfa4-2693-7af2-a94d-25dc0d4d433b-0', tool_calls=[{'name': 'customer_records', 'args': {'qu

In [12]:
try:
    response = agent.invoke({"messages":[{"role":"user","content":"here is my  api_key is sk-aphgxwvsyhuehhbhsgyzgyudg223"}]})
except Exception as e:
    print(f"Blocked as ecxcepted{e}")
    

In [13]:
response

{'messages': [HumanMessage(content='here is my  api_key is sk-aphgxwvsyhuehhbhsgyzgyudg223', additional_kwargs={}, response_metadata={}, id='a57af183-40b5-44a5-8868-3879f3d81983'),
  AIMessage(content="I can't help with that.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 231, 'total_tokens': 239, 'completion_time': 0.051805977, 'completion_tokens_details': None, 'prompt_time': 0.011610851, 'prompt_tokens_details': None, 'queue_time': 0.051532209, 'total_time': 0.063416828}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdfa4-2998-72e2-8355-9a7cfb4bd4b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 231, 'output_tokens': 8, 'total_tokens': 239})]}

In [14]:
## Human in the loop middleware 
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool
@tool
def search_web(query:str)-> str:
    """search the web for information"""
    return f"search results for :{query}"

@tool
def send_mail(to:str,subject:str,body:str)->str:
    """send an email to the receiptent"""
    return f"the emial succesfullt send to :{to}"

@tool
def delete_records(table:str,condition:str)->str:
    """delete records from the database"""
    return f"Deleted records from the table {table} where {condition}"

hit_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_web,send_mail,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_mail":True,
                "delete_records":True,
                "search_web":False
            }
        )
    ],
    checkpointer=InMemorySaver()
)

print("Agent succesfully created with human in the loop middleware")




Agent succesfully created with human in the loop middleware


In [15]:
config = {"configurable":{"thread_id":"1"}}

In [16]:
config = {"configurable":{"thread_id":"1"}}

result = hit_agent.invoke({"messages":[{"role":"user","content":"send an email to the john@email.com about project details"}]},config=config)
print("Agent paused awaiting for human approval")
print(result)

Agent paused awaiting for human approval
{'messages': [HumanMessage(content='send an email to the john@email.com about project details', additional_kwargs={}, response_metadata={}, id='36058601-df21-4b24-b5c5-2fe0ea95133d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dbbawzd9g', 'function': {'arguments': '{"body":"Hello John, this email contains the project details.","subject":"Project Details","to":"john@email.com"}', 'name': 'send_mail'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 347, 'total_tokens': 385, 'completion_time': 0.116652746, 'completion_tokens_details': None, 'prompt_time': 0.019446347, 'prompt_tokens_details': None, 'queue_time': 0.051908082, 'total_time': 0.136099093}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdfa4-2e58-78f3-83cf-91dc36

In [17]:
approved_result = hit_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)

print("Approved Final Response")
approved_result["messages"][-1].content

Approved Final Response


''

In [18]:
for i, msg in enumerate(approved_result["messages"]):
    print("="*50)
    print(i)
    print(type(msg).__name__)
    print(msg.content)

0
HumanMessage
send an email to the john@email.com about project details
1
AIMessage

2
ToolMessage
the emial succesfullt send to :john@email.com
3
AIMessage



In [19]:
config2 = {"configurable":{"thread_id":"2"}}

result = hit_agent.invoke({"messages":[{"role":"user","content":"delete the data from the database where active = false"}]},
                          config=config2)

print("waiting for approval..........")

approved_result = hit_agent.invoke(
    Command(resume={"decisions":[{"type":"reject","reason":"Too risky need a dma review"}]},
            ),
    config=config2
)

print("Rejected approval")
print(approved_result["messages"])

waiting for approval..........
Rejected approval
[HumanMessage(content='delete the data from the database where active = false', additional_kwargs={}, response_metadata={}, id='76542c0c-0719-46eb-9141-7682f9262301'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'w7hh7y0cx', 'function': {'arguments': '{"condition":"active = false","table":"database_table"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 346, 'total_tokens': 370, 'completion_time': 0.062367115, 'completion_tokens_details': None, 'prompt_time': 0.027781671, 'prompt_tokens_details': None, 'queue_time': 0.053122759, 'total_time': 0.090148786}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdfa4-392c-7502-a379-34355bbfcd23-0', tool_calls=[{'name': 'delete_records', 'args': {'c

In [20]:
## Custom Guardrail before agent hook

from typing import Any
from langchain.agents.middleware import AgentState,AgentMiddleware,hook_config
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.runtime import Runtime
from langchain.messages import HumanMessage

class ContentModifiedMiddleware(AgentMiddleware):
    """ Deterministic guardrail: used for blocking the content having banned keywords
    this runs before the process everything ;NO llm cost"""
    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in  banned_keywords]
        
@hook_config(can_jump_to=["end"])
def before_agent(self,state:AgentState,runtime:Runtime) -> dict[str,Any] | None:
    if not state["messages"]:
        return None
    
    first_message = state["messages"][0]
    if not isinstance(first_message, HumanMessage):
        return None
    
    content = first_message.content.lower()
    
    for keyword in self.banned_keywords:
        if keyword in content:
            print(f"Blocked keyword detected{keyword}")
            return{
            "messages":[
                {"role":"assistant",
                 "content":("i cannot process content containing inappropriate content,PLease rephrase your content")}
            ],
                "jump_to":"end"
        }
    return None

@tool
def search_tool(query:str)->str:
    """search in the internet"""
    return f"Results for {query}"

filtered_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_tool],
    middleware=[
        ContentModifiedMiddleware(
            banned_keywords=["hack","exploit","malware","jailbreak","bypass"]  
        )
    ]
)
print("Content filter agent created")    

Content filter agent created


In [21]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"what is machine learning"}]})
result["messages"][-1].content



'Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perform a specific task without using explicit instructions. Instead, the machine learns from the data it is given, identifying patterns and relationships within the data to make predictions, classify objects, or make decisions.\n\nThere are several types of machine learning, including:\n\n1. Supervised learning: The machine is trained on labeled data, where the correct output is already known.\n2. Unsupervised learning: The machine is trained on unlabeled data, and it must find patterns or relationships within the data on its own.\n3. Reinforcement learning: The machine learns by interacting with an environment and receiving feedback in the form of rewards or penalties.\n\nMachine learning has many applications, including:\n\n1. Image and speech recognition\n2. Natural language processing\n3. Predictive analytics\n4. Recommendation systems\n5. 

In [22]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"How to hack a system"}]})
result["messages"][-1].content


"I can't provide information or guidance on illegal or harmful activities. Is there something else I can help you with?"

In [23]:
from langchain_core.messages import HumanMessage

class ContentModifiedMiddleware(AgentMiddleware):

    def __init__(self, banned_keywords):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime):

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]

        if not isinstance(last_message, HumanMessage):
            return None

        content = last_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked keyword detected: {keyword}")

                return {
                    "messages": [
                        {
                            "role": "assistant",
                            "content": "I cannot process content containing inappropriate content. Please rephrase your request."
                        }
                    ],
                    "jump_to": "end",
                }

        return None

In [24]:
filtered_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_tool],
    middleware=[
        ContentModifiedMiddleware(
            banned_keywords=["hack","exploit","malware","jailbreak","bypass"]  
        )
    ]
)

In [25]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"How to hack a system"}]})
result["messages"][-1].content


Blocked keyword detected: hack


'I cannot process content containing inappropriate content. Please rephrase your request.'

In [26]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"what is machine learning"}]})
result["messages"][-1].content



'Machine learning is a type of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perform a specific task without using explicit instructions. Instead, the machine learns from data, identifying patterns and making decisions based on that data.\n\nThere are several types of machine learning, including:\n\n1. Supervised learning: The machine is trained on labeled data, where the correct output is already known.\n2. Unsupervised learning: The machine is trained on unlabeled data and must find patterns or structure in the data.\n3. Reinforcement learning: The machine learns by interacting with an environment and receiving rewards or penalties for its actions.\n\nMachine learning has many applications, including:\n\n1. Image and speech recognition\n2. Natural language processing\n3. Predictive modeling and forecasting\n4. Recommendation systems\n5. Autonomous vehicles\n\nSome of the key benefits of machine learning include:\n\n1. Im

In [31]:
## Csutom Guardrail after agent hook
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langchain_core.tools import tool
from langchain_groq import ChatGroq

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model based guardrail:use llm and to evaluate response safety.
    Runs AFTER the agent produces a response before it reaches to the user
    """
    def __init__(self):
        super().__init__()
        self.safety_model = ChatGroq(model = "llama-3.3-70b-versatile",temperature=0)
        
    @hook_config(can_jump_to=["end"])
    def after_agent(self, state:AgentState, runtime:Runtime):
        if not state["messages"]:
            return None
        
        last_message = state["messages"][-1]
        if last_message.type != 'ai':
            return None
        
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke(safety_prompt)
        print("Safety model verdict:", result.content)
        
        if "UNSAFE" in result.content.upper():
            print("Output falgged as unsafe-replacing with safe fallback")
            last_message.content = (
                "I am unable to provide that resposne "
                "please rephrase your text"
            )
        return None
    
@tool
def gerneral_tool(query:str)-> str:
    """ A general purpose tool"""
    return f"Tool Result {query}"
safe_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [gerneral_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("output safety succesfully done")

output safety succesfully done


In [32]:
result = safe_agent.invoke({"messages":[{"role":"user","content":"How to hack a system"}]})
result["messages"][-1].content

Safety model verdict: SAFE


"I can't help with that. Hacking into systems without permission is against the law and unethical. If you're interested in cybersecurity, I can provide information on how to learn about it legally and ethically."

In [33]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.runtime import Runtime


# ============================================================
# Custom Safety Guardrail Middleware
# ============================================================

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail.

    Evaluates the AI-generated response AFTER the agent produces it.
    If the response is unsafe, it replaces it with a safe fallback.
    """

    def __init__(self):
        super().__init__()

        self.safety_model = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0
        )

    @hook_config(can_jump_to=["end"])
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ):
        # Make sure messages exist
        if not state["messages"]:
            return None

        # Get the last message
        last_message = state["messages"][-1]

        # after_agent should evaluate the AI response
        if last_message.type != "ai":
            return None

        # Safety evaluation prompt
        safety_prompt = f"""
Evaluate whether the following AI response is safe and appropriate
for users.

Reply with ONLY one word:
SAFE
or
UNSAFE

AI response:
{last_message.content}
"""

        # ChatGroq expects a string or list of BaseMessages,
        # NOT {"messages": [...]}
        result = self.safety_model.invoke(safety_prompt)

        verdict = result.content.strip().upper()

        print("AI response:", last_message.content)
        print("Safety verdict:", verdict)

        # Replace unsafe response
        if "UNSAFE" in verdict:
            print("Output flagged as unsafe - replacing with safe fallback")

            last_message.content = (
                "I am unable to provide that response. "
                "Please rephrase your request."
            )

        return None


# ============================================================
# General Purpose Tool
# ============================================================

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool Result: {query}"


# ============================================================
# Create Agent
# ============================================================

safe_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[general_tool],
    middleware=[
        SafetyGuardrailMiddleware()
    ]
)

print("Safety guardrail agent created successfully!")


# ============================================================
# Test Agent
# ============================================================

result = safe_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How to hack a system"
        }
    ]
})


# ============================================================
# Final Response
# ============================================================

print("\nFinal Response:")
print(result["messages"][-1].content)

Safety guardrail agent created successfully!
AI response: I can't answer that.
Safety verdict: SAFE

Final Response:
I can't answer that.


In [34]:
result = safe_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Give me some information"
        }
    ]
})

AI response: I hope this helps. Let me know if you have any other questions.
Safety verdict: SAFE
